In [3]:
import urllib.request
import datetime
import os

# Створюємо папку для зберігання файлів
data_dir = "../vhi_data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

def download_noaa_data(province_id, year1=1981, year2=2024):
    """
    Функція для завантаження даних VHI по заданій області.
    """
    # 1. Захист від повторного завантаження (як вимагається в завданні)
    # Шукаємо файли, які починаються з потрібного ID
    existing_files = [f for f in os.listdir(data_dir) if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже завантажені. Пропускаємо.")
        return

    # 2. Формуємо URL
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean"

    # 3. Завантажуємо та зберігаємо
    try:
        with urllib.request.urlopen(url) as response:
            html = response.read()

        # Додаємо дату та час до імені файлу
        now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"vhi_id_{province_id}_{now}.csv"
        filepath = os.path.join(data_dir, filename)

        # Записуємо дані у файл
        with open(filepath, 'wb') as file:
            file.write(html)

        print(f"Успішно завантажено: {filename}")

    except Exception as e:
        print(f"Помилка при завантаженні області {province_id}: {e}")

# 4. Запускаємо завантаження для областей від 1 до 27
# provinceID=0 пропускаємо згідно з умовою
print("Починаємо завантаження даних...")
for i in range(1, 28):
    download_noaa_data(i)
print("Процес завершено!")

Починаємо завантаження даних...
Дані для області 1 вже завантажені. Пропускаємо.
Дані для області 2 вже завантажені. Пропускаємо.
Дані для області 3 вже завантажені. Пропускаємо.
Дані для області 4 вже завантажені. Пропускаємо.
Дані для області 5 вже завантажені. Пропускаємо.
Дані для області 6 вже завантажені. Пропускаємо.
Дані для області 7 вже завантажені. Пропускаємо.
Дані для області 8 вже завантажені. Пропускаємо.
Дані для області 9 вже завантажені. Пропускаємо.
Дані для області 10 вже завантажені. Пропускаємо.
Дані для області 11 вже завантажені. Пропускаємо.
Дані для області 12 вже завантажені. Пропускаємо.
Дані для області 13 вже завантажені. Пропускаємо.
Дані для області 14 вже завантажені. Пропускаємо.
Дані для області 15 вже завантажені. Пропускаємо.
Дані для області 16 вже завантажені. Пропускаємо.
Дані для області 17 вже завантажені. Пропускаємо.
Дані для області 18 вже завантажені. Пропускаємо.
Дані для області 19 вже завантажені. Пропускаємо.
Дані для області 20 вже зав

In [4]:
import pandas as pd
import os

data_dir = "../vhi_data"

# 1. Словник для заміни індексів (NOAA -> Українська абетка)
# Ключ - старий індекс, значення - новий індекс
index_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

# 2. Словник з назвами областей (за новими індексами)
province_names = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
    6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
    11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
    16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "АР Крим",
    26: "м. Київ", 27: "м. Севастополь"
}

def create_cleaned_dataframe(directory):
    all_dataframes = []

    # Задаємо назви колонок. 'empty' потрібен як фіктивна колонка для зчитування "висячих" ком
    # в кінці рядків датасету NOAA. Після зчитування цей мусор видаляємо.
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']

    # Перебираємо всі завантажені файли
    for filename in os.listdir(directory):
        if not filename.endswith(".csv"):
            continue

        filepath = os.path.join(directory, filename)

        # Витягуємо старий ID області з імені файлу (наприклад, з vhi_id_8_2024...)
        old_id = int(filename.split('_')[2])
        new_id = index_mapping[old_id]

        # Зчитуємо файл.
        # header=1 пропускає рядок з тегом <pre>, skipfooter=1 відкидає </pre> в кінці.
        # Використовуємо sep=',' та skipinitialspace=True для правильного зчитування пробілів.
        df = pd.read_csv(filepath, header=1, names=headers, skipfooter=1, engine='python', sep=',', skipinitialspace=True)

        # Видаляємо фіктивну колонку
        # Додали 'empty', бо в кінці кожного рядка у файлах NOAA стоїть зайва кома.
        # Через неї Pandas бачить 8 колонок замість 7 і видає помилку.
        # Зчитуємо цей "сміттєвий" стовпець і одразу видаляємо.
        if 'empty' in df.columns:
            df = df.drop(columns=['empty'])

        # Очищення від залишків HTML (відкидаємо рядки, де в колонці Year є теги)
        df = df[~df['Year'].astype(str).str.contains('<')]

        # Конвертуємо колонки у числові типи (помилки перетворяться на NaN)
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df['Week'] = pd.to_numeric(df['Week'], errors='coerce')
        df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')

        # Видаляємо рядки з відсутніми ключовими даними
        df = df.dropna(subset=['Year', 'Week', 'VHI'])

        # Відкидаємо некоректні значення VHI (наприклад, -1)
        df = df[df['VHI'] >= 0]

        # Додаємо колонки з новим індексом та назвою області
        df['Province_ID'] = new_id
        df['Province_Name'] = province_names[new_id]

        all_dataframes.append(df)

    # Об'єднуємо всі таблиці в одну велику
    full_df = pd.concat(all_dataframes, ignore_index=True)

    # Міняємо порядок колонок для зручності
    full_df = full_df[['Year', 'Week', 'Province_ID', 'Province_Name', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

    return full_df

# Викликаємо функцію
df = create_cleaned_dataframe(data_dir)

print("Дані успішно очищено та об'єднано!")
# Виводимо перші 5 рядків
df.head()

Дані успішно очищено та об'єднано!


,Year,Week,Province_ID,Province_Name,SMN,SMT,VCI,TCI,VHI
0,1982,2,21,Хмельницька,0.063,261.53,55.89,38.20,47.04
1,1982,3,21,Хмельницька,0.063,263.45,57.30,32.69,44.99
2,1982,4,21,Хмельницька,0.061,265.10,53.96,28.62,41.29
3,1982,5,21,Хмельницька,0.058,266.42,46.87,28.57,37.72
4,1982,6,21,Хмельницька,0.056,267.47,39.55,30.27,34.91


In [5]:
def get_vhi_for_province_and_year(dataframe, province_id, year):
    """
    Повертає ряд VHI (тижні та значення) для вказаної області та року.
    """
    # Фільтруємо таблицю за двома умовами одночасно
    filtered_df = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]

    # Залишаємо лише колонки з тижнем та VHI для зручності,
    # і скидаємо старі індекси
    result = filtered_df[['Week', 'VHI']].reset_index(drop=True)

    return result

# перевырка роботи ф-ції
# Наприклад, візьмемо Вінницьку область (ID 1) і 2020 рік
province_to_search = 1
year_to_search = 2020

vhi_series = get_vhi_for_province_and_year(df, province_to_search, year_to_search)

print(f"Ряд VHI для області з ID {province_to_search} за {year_to_search} рік:")
vhi_series.head(10) # Виведемо перші 10 тижнів

Ряд VHI для області з ID 1 за 2020 рік:


,Week,VHI
0,1,40.92
1,2,43.19
2,3,44.74
3,4,45.29
4,5,44.80
5,6,43.92
6,7,43.10
7,8,42.88
8,9,43.71
9,10,45.61


In [6]:
def get_vhi_statistics(dataframe, province_id, year):
    """
    Шукає екстремуми (min, max), середнє та медіану VHI
    для вказаної області та року.
    """
    # Фільтруємо дані
    filtered_df = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]

    # Якщо даних за цей рік немає, повертаємо порожній результат
    if filtered_df.empty:
        return "Немає даних для вибраної області та року."

    # Обчислюємо статистику
    vhi_min = filtered_df['VHI'].min()
    vhi_max = filtered_df['VHI'].max()
    vhi_mean = filtered_df['VHI'].mean()
    vhi_median = filtered_df['VHI'].median()

    # Формуємо красивий результат у вигляді словника
    result = {
        'Province_ID': province_id,
        'Year': year,
        'VHI_Min': vhi_min,
        'VHI_Max': vhi_max,
        'VHI_Mean': round(vhi_mean, 2),
        'VHI_Median': vhi_median
    }

    return result

# Перевырка фу-ії
# Візьмемо Київську область (ID 9) і 2010 рік
province_to_search = 9
year_to_search = 2010

stats = get_vhi_statistics(df, province_to_search, year_to_search)

print(f"Статистика VHI для області з ID {province_to_search} за {year_to_search} рік:")
for key, value in stats.items():
    print(f"{key}: {value}")

Статистика VHI для області з ID 9 за 2010 рік:
Province_ID: 9
Year: 2010
VHI_Min: 26.82
VHI_Max: 52.79
VHI_Mean: 40.7
VHI_Median: 40.705
